In [ ]:
from nanogpt.layers import MultiHeadAttention, Embedding, FeedForward, LayerNorm
from nanogpt import Transformer
from nanogpt import Tokenizer, Indicer
from nanogpt import SlidingWindow
from nanogpt.loss_functions import CrossEntropyLoss
from nanogpt import Optimizer
import numpy as np
import os

## Data Loading & Preprocessing

In [ ]:
path_to_text = "data/input.txt"
with open(path_to_text, 'r') as f:
    text = f.read()

def preprocess_text(text):
    text = text.lower()
    punctuation = ['.', ',', '!', '?', ';', ':', '"', "'", '(', ')', '[', ']', '{', '}', '-', '_', '/', '\\']
    for char in punctuation:
        text = text.replace(char, ' ' + char + ' ')
    return text

processed_text = preprocess_text(text)
print("Processed text:", processed_text[:100])

## Tokenization & Encoding

In [ ]:
tokenizer = Tokenizer(vocab_size=10000)
token_map = tokenizer.sequentialize(processed_text)
indicer = Indicer(vocab_size=10000)
indicer.fit(list(token_map))
encoded_tokens = indicer.encode(list(token_map))
vocab_size = len(set(processed_text.split()))
print("Encoded tokens:", encoded_tokens[:10])
print("Decoded text:", indicer.decode(encoded_tokens[:10]))
print("Vocabulary size:", vocab_size)
print("Total tokens:", len(encoded_tokens))

In [ ]:
split_index = int(len(token_map) * 0.8)
train_token_map = token_map[:split_index]
test_token_map = token_map[split_index:]
encoded_train_tokens = indicer.encode(list(train_token_map))
encoded_test_tokens = indicer.encode(list(test_token_map))
print("Train tokens:", len(train_token_map))
print("Test tokens:", len(test_token_map))

## Sanity Checks

In [ ]:
sliding_window = SlidingWindow(train_token_map)
sample = sliding_window.sample()
encoded_sample = indicer.encode(list(sample))
print("Sample shape:", np.array(encoded_sample).shape)

In [ ]:
embedding = Embedding(vocab_size, 768)
embedding.forward(encoded_sample).data

In [ ]:
transformer = Transformer(vocab_size, 768, 4, 3072, 4)
output = transformer.forward(encoded_sample)
print("Output shape:", output.data.shape)
print("Embedding weights shape:", transformer.embedding.weights.data.shape)
print("Projection weights shape:", transformer.projection.weights.data.shape)

## Training

In [ ]:
def get_batch(data, block_size=256):
    i = np.random.randint(0, len(data) - block_size - 1)
    x = data[i : i + block_size]
    y = data[i + 1 : i + block_size + 1]
    return x, y

def visit(node, visited=None):
    """Check for vanishing gradients in the computation graph."""
    if visited is None:
        visited = set()
    if id(node) in visited:
        return
    visited.add(id(node))
    for child in node._children:
        visit(child, visited)
        if not np.any(child.grad):
            print("VANISHING GRADIENT FOUND")
            return

In [ ]:
def train_loop(model, data, iters=100, lr=0.001):
    loss_func = CrossEntropyLoss()
    optimizer = Optimizer(loss_func, model)
    for i in range(iters):
        optimizer.zero_grad()
        x, y = get_batch(data)
        output = model.forward(x)
        loss = loss_func.forward(output, y)
        loss.backward()
        optimizer.step(lr)
        visit(loss)
        print(f"Iteration {i}: loss={loss.data:.4f}")

In [ ]:
model = Transformer(vocab_size, 768, 2, 3072, 2)
train_loop(model, encoded_train_tokens, iters=100)

## Text Generation

In [ ]:
def generate(model, tokens, indicer, max_tokens=20, temperature=0.7):
    encoded_input = indicer.encode(list(tokens))
    text = list(encoded_input)
    for _ in range(max_tokens):
        output = model.forward(text).data
        logits = output[-1]
        scaled_logits = logits / temperature
        scaled_logits -= scaled_logits.max()  # numerical stability
        exps = np.exp(scaled_logits)
        probabilities = exps / np.sum(exps)
        token = np.random.choice(len(probabilities), p=probabilities)
        text.append(token)
    return text

In [ ]:
prompt = "to be or not to be"
processed_prompt = preprocess_text(prompt)
processed_prompt = tokenizer.sequentialize(processed_prompt)

generated_tokens = generate(model, processed_prompt, indicer)
decoded_text = indicer.decode(generated_tokens)
print(" ".join(decoded_text))